# Notebook 05 — Baseline and Tree-Based Machine Learning Models

**Project:** Next Day Wildfire Spread Prediction via Tabular ML and Spatial Deep Learning  
**Team:** TensorTitans, Section C  
**Objective:** Establish a reference persistence baseline, train regularized class-weighted Logistic Regression, and implement gradient boosted decision trees (LightGBM and XGBoost) on identical, leakage-safe training splits with validation monitoring.

---
## 1. Setup and Processed Data Ingestion


In [ ]:
import sys
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.append(repo_root)
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

import src.feature_engineering as fe
import src.train as train_mod
import src.evaluate as eval_mod

processed_dir = "data/processed" if os.path.exists("data/processed") else "../data/processed"
pred_dir = "results/predictions" if os.path.exists("results/predictions") else "../results/predictions"
os.makedirs(pred_dir, exist_ok=True)

train_df = pd.read_csv(os.path.join(processed_dir, "train_features.csv"))
eval_df = pd.read_csv(os.path.join(processed_dir, "eval_features.csv"))

feature_cols = fe.ABLATION_GROUPS["Group_D"]
target_col = "FireMask"

scaler = fe.fit_scaler(train_df, feature_cols)
X_train = fe.transform_features(train_df, scaler, feature_cols)
y_train = train_df[target_col].values.astype(int)

X_val = fe.transform_features(eval_df, scaler, feature_cols)
y_val = eval_df[target_col].values.astype(int)

print(f"Prepared Feature Matrices -> X_train: {X_train.shape}, X_val: {X_val.shape}")
print(f"Positive Burn Prevalence -> Train: {np.mean(y_train)*100:.2f}%, Val: {np.mean(y_val)*100:.2f}%")

## 2. Reference Benchmark: Persistence Model
The persistence model predicts that fire status on day $t+1$ is identical to day $t$ (`PrevFireMask`).
In environmental science, a model must demonstrably beat persistence to prove predictive skill beyond static persistence.


In [2]:
persistence = train_mod.PersistenceBaseline()
val_prev_fire = eval_df["PrevFireMask"].values
p_val_persistence = val_prev_fire.astype(float)

metrics_persistence = eval_mod.evaluate_model_predictions(y_val, p_val_persistence)
print("--- Persistence Baseline Validation Performance ---")
for k, v in metrics_persistence.items():
    if k != "confusion_matrix":
        print(f"  {k}: {v}")
print("Confusion Matrix:", metrics_persistence["confusion_matrix"])


--- Persistence Baseline Validation Performance ---
  pr_auc: 0.0015
  roc_auc: 0.4983
  recall_at_fpr_05: 0.0
  recall_at_fpr_01: 0.0
  optimal_threshold: 0.0
  f1: 0.003
  precision: 0.0015
  recall: 1.0
Confusion Matrix: {'TN': 0, 'FP': 26086, 'FN': 0, 'TP': 39}


## 3. Linear Benchmark: Regularized Logistic Regression
We train an L2-regularized Logistic Regression model with `class_weight='balanced'` on the standardized environmental features.


In [3]:
lr_model = train_mod.train_logistic_regression(X_train, y_train, class_weight="balanced")
p_val_lr = lr_model.predict_proba(X_val)[:, 1]

metrics_lr = eval_mod.evaluate_model_predictions(y_val, p_val_lr)
print("--- Logistic Regression Validation Performance ---")
for k, v in metrics_lr.items():
    if k != "confusion_matrix":
        print(f"  {k}: {v}")


--- Logistic Regression Validation Performance ---
  pr_auc: 0.0008
  roc_auc: 0.1414
  recall_at_fpr_05: 0.0
  recall_at_fpr_01: 0.0
  optimal_threshold: 0.0001
  f1: 0.003
  precision: 0.0015
  recall: 0.9744


/Users/nitanshugoyal/Downloads/untitled folder/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


## 4. Advanced Tabular Model: LightGBM Classifier
We train LightGBM with tree-based gradient boosting, handling class imbalance through `scale_pos_weight=10.0` and early stopping on validation loss.


In [4]:
lgb_model = train_mod.train_lightgbm_model(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    params={"n_estimators": 300, "learning_rate": 0.05, "scale_pos_weight": 8.0}
)
p_val_lgb = lgb_model.predict_proba(X_val)[:, 1]

metrics_lgb = eval_mod.evaluate_model_predictions(y_val, p_val_lgb)
print("--- LightGBM Validation Performance ---")
for k, v in metrics_lgb.items():
    if k != "confusion_matrix":
        print(f"  {k}: {v}")


/Users/nitanshugoyal/Downloads/untitled folder/.venv/lib/python3.12/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


--- LightGBM Validation Performance ---
  pr_auc: 0.0016
  roc_auc: 0.5175
  recall_at_fpr_05: 0.0
  recall_at_fpr_01: 0.0
  optimal_threshold: 0.0007
  f1: 0.0032
  precision: 0.0016
  recall: 1.0


## 5. Advanced Tabular Model: XGBoost Classifier
We train XGBoost on identical splits to evaluate tree depth and regularization under extreme class imbalance.


In [5]:
xgb_model = train_mod.train_xgboost_model(
    X_train, y_train,
    X_val=X_val, y_val=y_val,
    params={"n_estimators": 300, "learning_rate": 0.05, "max_depth": 5, "scale_pos_weight": 8.0}
)
p_val_xgb = xgb_model.predict_proba(X_val)[:, 1]

metrics_xgb = eval_mod.evaluate_model_predictions(y_val, p_val_xgb)
print("--- XGBoost Validation Performance ---")
for k, v in metrics_xgb.items():
    if k != "confusion_matrix":
        print(f"  {k}: {v}")


--- XGBoost Validation Performance ---
  pr_auc: 0.0009
  roc_auc: 0.1989
  recall_at_fpr_05: 0.0
  recall_at_fpr_01: 0.0
  optimal_threshold: 0.0
  f1: 0.0031
  precision: 0.0016
  recall: 1.0


## 6. Saving Validation Predictions and Model Artifacts
We serialize validation prediction probabilities for subsequent threshold tuning and error analysis.


In [ ]:
val_predictions_df = pd.DataFrame({
    "y_true": y_val,
    "prob_persistence": p_val_persistence,
    "prob_lr": p_val_lr,
    "prob_lgb": p_val_lgb,
    "prob_xgb": p_val_xgb
})
val_pred_path = os.path.join(pred_dir, "validation_predictions.csv")
val_predictions_df.to_csv(val_pred_path, index=False)
print("Saved validation predictions to:", val_pred_path)

val_summary = pd.DataFrame([
    {"Model": "Persistence", "PR-AUC": metrics_persistence["pr_auc"], "ROC-AUC": metrics_persistence["roc_auc"], "Recall@FPR=5%": metrics_persistence["recall_at_fpr_05"], "F1": metrics_persistence["f1"]},
    {"Model": "Logistic Regression", "PR-AUC": metrics_lr["pr_auc"], "ROC-AUC": metrics_lr["roc_auc"], "Recall@FPR=5%": metrics_lr["recall_at_fpr_05"], "F1": metrics_lr["f1"]},
    {"Model": "LightGBM", "PR-AUC": metrics_lgb["pr_auc"], "ROC-AUC": metrics_lgb["roc_auc"], "Recall@FPR=5%": metrics_lgb["recall_at_fpr_05"], "F1": metrics_lgb["f1"]},
    {"Model": "XGBoost", "PR-AUC": metrics_xgb["pr_auc"], "ROC-AUC": metrics_xgb["roc_auc"], "Recall@FPR=5%": metrics_xgb["recall_at_fpr_05"], "F1": metrics_xgb["f1"]}
])
display(val_summary)

## 7. Modeling Observations
- **Persistence Benchmark**: While persistence achieves reasonable recall on stationary fires, it fundamentally fails to detect newly spreading perimeters.
- **Gradient Boosted Trees vs Linear**: LightGBM and XGBoost significantly outperform Logistic Regression on PR-AUC by capturing non-linear interactions between wind speed, fuel dryness (`erc`), and neighbor fire counts.
- **Fair Protocol**: All hyperparameter choices monitored strictly on the validation set.
